In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
archivo = "/content/drive/MyDrive/Ciencia De Datos/conteo_palabras_video_youtube.xlsx"
completo = pd.read_excel(archivo, sheet_name="Conteo completo", header=1)
depurado = pd.read_excel(archivo, sheet_name="Conteo depurado", header=1)
excluidas = pd.read_excel(archivo, sheet_name="Palabras excluidas", header=1)

/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [ ]:
for df in (completo, depurado, excluidas):
  df.columns = [str(c).strip() for c in df.columns]

completo = completo.dropna(subset=["Palabra", "Frecuencia"]).copy()
depurado = depurado.dropna(subset=["Palabra", "Frecuencia"]).copy()
completo["Frecuencia"]= completo["Frecuencia"].astype(int)
depurado["Frecuencia"]= depurado["Frecuencia"].astype(int)

print(f'Palabras totales: {completo["Frecuencia"].sum():,}')
print(f'Palabras unicas: {len(completo)}')
print(f'Palabras totales depuradas: {depurado["Frecuencia"].sum():,}')
print(f'Palabras unicas depradas: {len(depurado)}')
display(completo.head(10))

Palabras totales: 4,437
Palabras unicas: 1154
Palabras totales depuradas: 2,344
Palabras unicas depradas: 1063


,Palabra,Frecuencia
0,que,309
1,de,215
2,a,108
3,la,106
4,no,97
5,y,97
6,el,86
7,lo,84
8,es,71
9,en,64


**Medidas de tendencia central**

|Medida | Definicion | Utilidad |
|---|---|---|
|Media| Suma de frecuencias divida entre palabras unicas|Frecuencia promedio, sensible a plabras extremadamente repetidas|
|Mediana| Valor central de las frecuencias ordenadas | Describe mejor una dsitribucion no simetrica|
|Moda| Frecuencia que ocurre en mas palabras|Mostrar muchos palabras que aparecen una vez|
|Cuartiles|Dividen los datos ordenados en 4 partes|Localizar el 25%, 50% y 75% de las frecuencias|
|Varianza| Promedio de desviaciones cuadraticas con respecto a la media| Cualifica la dispercion|
|Desviacion Estandar| raiz de la varianza | Dispersion en las mismas unidades de frecuencia|
|IQR - Rango Intercuartilico| Q3 - Q1 | Mide una dispersion mas robusta del 50% central|
|Asimetria| Mide la falta de simetria| Un valor alto revela una cola de palabras muy frecuentes|


In [ ]:
def gini(valores):
  x=np.sort(np.asarray(valores,dtype=float))
  if len(x)==0 or x.sum()==0:
    return np.nan
  n=len(x)
  return(2*np.sum(np.arange(1,n+1)*x)/(n*x.sum())) - (n+1)/n

def entropia(valores):
  x=np.asarray(valores,dtype=float)
  p=p/p.sum()
  h=-(p*np.log2(p)).sum()
  return h/np.log2(len(p)) if len(p)>1 else 0.0

In [ ]:
def resumen_estadistico(df,nombre):
  f = df["Frecuencia"]
  modos = f.mode().tolist()
  return pd.series({
      "Corpus":nombre,
      "Palabras totales (tokens)":int(f.sum()),
      "Palabras unicas":int(f.size),
      "Media":f.mean(),
      "Mediana":f.median(),
      "Moda":", ".join(map(str,modos)),
      "Minimo":int(f.min()),
      "Maximo":int(f.max()),
      "Q1 (25%)":int(f.quantile(.25)),
      "Q2 (50%)":int(f.quantile(.5)),
      "Q3 (75%)":int(f.quantile(.75)),
      "Rango":int(f.max()-f.min()),
      "Varianza muestral":f.var(ddof=1),
      "Desviacion Estandar Muestral":f.std(ddof=1),
      "Asimetria":f.skew(),
      "Curtosis(exceso)":f.kurt(),
      "Diversidad lexica":f.size/f.sum(),
      "Entropia":entropia(f),
      "Gini":gini(f)
  })

  resumen=pd.DataFrame([
      resumen_estadistico(completo,"Completo"),
      resumen_estadistico(depurado,"Depurado"),
      resumen_estadistico(excluidas,"Excluidas")
  ]).set_index("Corpus")
  display(resumen)